# Study area map

Report figure: (a) location of the study area in New Zealand, (b) the Napier–Hastings floodplain on a pre-event Sentinel-2 composite with the study AOI from `layer_config.py`.

Output: `outputs/study_area.png` (300 dpi).

## 1. Setup

In [ ]:
%matplotlib inline
import sys, io, math, importlib
from pathlib import Path
from urllib.request import urlopen

import ee
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.ticker import FuncFormatter
from PIL import Image

_root = Path.cwd()
_nb = _root / "notebooks" if (_root / "notebooks" / "layer_config.py").exists() else _root
sys.path.insert(0, str(_nb))
import layer_config as cfg
importlib.reload(cfg)
_repo = _root if (_root / "processing" / "ee_auth.py").exists() else _root.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from processing.ee_auth import initialize_ee
from processing.map_furniture import add_scale_bar, add_north_arrow

initialize_ee()

OUT_DIR = _repo / "outputs"
OUT_DIR.mkdir(exist_ok=True)

W, S, E, N = cfg.AOI_BOUNDS
lat0, lon0 = cfg.MAP_CENTER
KM_PER_DEG_LON = 111.32 * math.cos(math.radians(lat0))

# Context panel: AOI centre ± CONTEXT_KM (wide enough to show Napier, Hastings and the coast)
CONTEXT_KM = 18
dlat = CONTEXT_KM / 111.32
dlon = CONTEXT_KM / KM_PER_DEG_LON
CTX = [lon0 - dlon, lat0 - dlat, lon0 + dlon, lat0 + dlat]   # W, S, E, N
print("AOI:", [round(v, 4) for v in cfg.AOI_BOUNDS], "| context:", [round(v, 4) for v in CTX])


## 2. Data: pre-event Sentinel-2, NZ outline, Hawke's Bay region

In [ ]:
# Pre-event Sentinel-2 true-colour composite for the context panel (summer before Gabrielle)
ctx_geom = ee.Geometry.Rectangle(CTX)

def _mask_s2_clouds(img):
    scl = img.select("SCL")
    clear = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(clear)

s2_pre = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(ctx_geom)
          .filterDate("2022-11-01", "2023-01-31")
          .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
          .map(_mask_s2_clouds)
          .median()
          .select(["B4", "B3", "B2"]))
print("S2 scenes used:", ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(ctx_geom)
      .filterDate("2022-11-01", "2023-01-31").filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30)).size().getInfo())

url = s2_pre.visualize(min=0, max=2200, gamma=1.2).getThumbURL(
    {"region": ctx_geom, "dimensions": 1600, "format": "png", "crs": "EPSG:4326"})
ctx_rgb = np.array(Image.open(io.BytesIO(urlopen(url).read())).convert("RGB"))
print("Context image:", ctx_rgb.shape)

# New Zealand outline (LSIB) and Hawke's Bay region (FAO GAUL), simplified for a small inset
def _fc_to_gdf(fc):
    return gpd.GeoDataFrame.from_features(fc.getInfo()["features"], crs="EPSG:4326")

nz_box = ee.Geometry.Rectangle([165.5, -47.6, 179.0, -34.0])
nz = _fc_to_gdf(ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
                .filter(ee.Filter.eq("country_na", "New Zealand"))
                .map(lambda f: f.setGeometry(f.geometry().intersection(nz_box, 1000).simplify(1000))))
hb = _fc_to_gdf(ee.FeatureCollection("FAO/GAUL/2015/level1")
                .filter(ee.Filter.eq("ADM0_NAME", "New Zealand"))
                .filter(ee.Filter.stringContains("ADM1_NAME", "Hawke"))
                .map(lambda f: f.simplify(1000)))
print("NZ features:", len(nz), "| Hawke's Bay features:", len(hb), hb.get("ADM1_NAME", []).tolist() if len(hb) else "")


## 3. Figure

In [ ]:
# Study area figure: (a) location in New Zealand, (b) context around the AOI
TOWNS = {  # approximate town centres (lat, lon)
    "Napier": (-39.4928, 176.9120),
    "Taradale": (-39.5370, 176.8460),
    "Hastings": (-39.6390, 176.8440),
    "Clive": (-39.5860, 176.9180),
}
RED = "#d7191c"
fmt_lon = FuncFormatter(lambda v, _: f"{v:.2f}°E")
fmt_lat = FuncFormatter(lambda v, _: f"{abs(v):.2f}°S")

fig = plt.figure(figsize=(11, 6.2))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 2.1], wspace=0.08)
ax_nz = fig.add_subplot(gs[0])
ax = fig.add_subplot(gs[1])

# (a) New Zealand locator
nz.plot(ax=ax_nz, color="#e0e0e0", edgecolor="#555555", linewidth=0.5)
if len(hb):
    hb.plot(ax=ax_nz, color="#9ecae1", edgecolor="#3182bd", linewidth=0.6)
ax_nz.add_patch(Rectangle((CTX[0], CTX[1]), CTX[2] - CTX[0], CTX[3] - CTX[1],
                          fill=False, ec=RED, lw=1.5, zorder=5))
ax_nz.annotate("Panel (b)", xy=(CTX[2], lat0), xytext=(CTX[2] + 1.2, lat0 - 1.0),
               fontsize=8, color=RED, arrowprops=dict(arrowstyle="-", color=RED, lw=0.8))
if len(hb):
    ax_nz.text(176.3, -38.9, "Hawke's Bay", fontsize=8, color="#08519c", ha="right")
ax_nz.set_xlim(166, 179)
ax_nz.set_ylim(-47.5, -34)
ax_nz.set_aspect(1 / math.cos(math.radians(-41)))
ax_nz.set_facecolor("#f4f8fb")
ax_nz.set_title("(a) Location in New Zealand", fontsize=11, loc="left")
ax_nz.set_xticks([168, 172, 176]); ax_nz.set_yticks([-46, -42, -38])
ax_nz.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0f}°E"))
ax_nz.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{abs(v):.0f}°S"))
ax_nz.tick_params(labelsize=8)
ax_nz.set_xlabel(""); ax_nz.set_ylabel("")
add_north_arrow(ax_nz, loc="upper left", size=20)

# (b) Context: pre-event Sentinel-2 with AOI and towns
ax.imshow(ctx_rgb, extent=[CTX[0], CTX[2], CTX[1], CTX[3]], origin="upper")
ax.add_patch(Rectangle((W, S), E - W, N - S, fill=False, ec=RED, lw=2.0, zorder=5))
ax.text(E + 0.004, N, f"Study AOI\n({2 * cfg.AOI_RADIUS_KM:.0f} × {2 * cfg.AOI_RADIUS_KM:.0f} km)",
        color=RED, fontsize=9, fontweight="bold", va="top", ha="left",
        path_effects=[plt.matplotlib.patheffects.withStroke(linewidth=2.5, foreground="white")])
for name, (la, lo) in TOWNS.items():
    if CTX[0] < lo < CTX[2] and CTX[1] < la < CTX[3]:
        ax.plot(lo, la, "o", ms=4, mfc="white", mec="black", mew=0.8, zorder=6)
        ax.text(lo + 0.006, la + 0.004, name, fontsize=9, zorder=6,
                path_effects=[plt.matplotlib.patheffects.withStroke(linewidth=2.5, foreground="white")])
ax.text(177.02, -39.47, "Hawke Bay", fontsize=10, style="italic", color="white", ha="center")
ax.set_xlim(CTX[0], CTX[2]); ax.set_ylim(CTX[1], CTX[3])
ax.set_aspect(1 / math.cos(math.radians(lat0)))
ax.xaxis.set_major_formatter(fmt_lon); ax.yaxis.set_major_formatter(fmt_lat)
ax.yaxis.tick_right()
ax.tick_params(labelsize=8)
ax.set_xlabel(""); ax.set_ylabel("")
ax.locator_params(nbins=4)
ax.set_title("(b) Napier–Hastings floodplain, Sentinel-2 Nov 2022–Jan 2023", fontsize=11, loc="left")
add_scale_bar(ax, KM_PER_DEG_LON, loc="lower left")
add_north_arrow(ax, loc="upper left")

fig.savefig(OUT_DIR / "study_area.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved", OUT_DIR / "study_area.png")
